# Aula 08 · Eliminação de Gauss

Esta aula apresenta o [capítulo 8 do site](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/). A ideia central: **várias incógnitas ligadas linearmente formam um sistema A x = b**, e o escalonamento da escola, organizado em três laços, resolve sistemas de qualquer tamanho.

**Ao fim da aula você consegue:**

1. retomar o circuito montado na aula 07 e ver o que o `np.linalg.solve` faz por dentro;
2. deduzir e programar a eliminação de Gauss e a substituição de trás para frente;
3. reconhecer o problema do pivô pequeno e resolvê-lo trocando linhas;
4. conferir uma solução substituindo-a no sistema, e desconfiar de sistemas mal condicionados.

**Roteiro:** 🧩 · 1. matrizes · 2. 🧑‍🏫 eliminação · 3. pivotamento · 4. confira · 5. mal condicionados · 6. outra área · 🎯 prática · 🧩 o celular · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Telecomunicações — onde está o celular?**
>
> *Numa ligação de emergência, a operadora precisa informar ao resgate a posição do
> aparelho. Três torres recebem o sinal, e o tempo que ele leva para ir e voltar dá
> a distância do aparelho a cada torre: 1,921 km, 2,773 km e 2,343 km. As torres
> estão em (0, 0), (4, 0) e (0, 3) km. "**Onde está a pessoa?**"*

Cada distância é um círculo em volta da torre, e o celular está onde os três se
cruzam. As equações de círculo não são lineares, mas **a diferença** entre duas
delas é. No fim da aula, você transforma o problema num sistema 2 × 2.

## 1. Sistemas lineares e matrizes

O circuito de três malhas que você montou a partir do desenho na aula 07. Até aqui,
o `np.linalg.solve` resolvia o sistema como uma caixa preta; nesta aula, você abre a
caixa. Lembrete da aula 07: `A[i, j]` é o elemento da linha `i`, coluna `j`, e `A[i]`
é a linha `i` inteira.

📖 [capítulo 8 · Sistemas lineares e matrizes](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#sistemas-lineares-e-matrizes)

In [ ]:
# 📦 dados prontos — só rode esta célula
# Correntes de malha de um circuito (leis de Kirchhoff): A x = b
A = np.array([[15.0, -5.0, 0.0],
              [-5.0, 20.0, -10.0],
              [0.0, -10.0, 25.0]])
b = np.array([10.0, 0.0, 5.0])

**✍️ Passo 1.** Imprima `A[1, 1]`, `A[2]` e `A[0, 2]`.

In [ ]:
# ✍️ passo 1

**Preveja:** quanto vale `A[0, 2]`, e o que ele significa no circuito?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`20.0`, `[0. -10. 25.]` e `0.0`. O zero diz que a malha 0 não divide resistor com a malha 2: elas não são vizinhas.

</details>

> 🧰 **Comando novo: `.copy()`**
>
> `b2 = b` **não** copia um array: dá a ele um segundo nome, e mexer em `b2` muda `b`.
> `b2 = b.copy()` copia de verdade. A eliminação de Gauss modifica a matriz, e por isso
> trabalha numa cópia.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
a = np.array([1.0, 2.0, 3.0])
outro_nome = a
outro_nome[0] = 99
print(a)

**✍️ Passo 2.** Faça `A2 = A.copy()`, mude `A2[0, 0] = 0` e imprima `A[0, 0]`.

In [ ]:
# ✍️ passo 2

**Preveja:** o `A` original mudou?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: continua `15.0`. Sem o `.copy()`, teria mudado — e a próxima célula que
usasse `A` receberia a matriz estragada, sem aviso nenhum.

📖 [capítulo 8 · Sistemas lineares e matrizes](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#sistemas-lineares-e-matrizes)

</details>

## 2. No quadro: a eliminação de Gauss

📖 [capítulo 8 · No quadro: a eliminação de Gauss](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#no-quadro-a-eliminacao-de-gauss)

### 🧑‍🏫 No quadro — eliminação e substituição

Caderno de papel aberto. No quadro:

1. o pivô $A_{00}$; o fator $A_{i0}/A_{00}$ de cada linha de baixo;
2. linha $i$ menos fator vezes linha 0 (em $A$ e em $b$): zera a coluna 0;
3. repetir com as colunas seguintes: matriz **triangular superior**;
4. substituição de trás para frente;
5. o circuito 3 × 3 inteiro, à mão.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

Eliminação: para cada coluna $k$ e cada linha $i > k$, com $\text{fator} = A_{ik}/A_{kk}$:
$A_{ij} \leftarrow A_{ij} - \text{fator}\cdot A_{kj}$ e $b_i \leftarrow b_i - \text{fator}\cdot b_k$.

Substituição: $x_i = \dfrac{b_i - \sum_{j > i} A_{ij}x_j}{A_{ii}}$, de $i = n-1$ até $0$.

</details>

**✍️ Passo 3.** Com `U = A.copy()` e `c = b.copy()`, zere a coluna 0 **só da linha 1**: calcule `fator = U[1, 0] / U[0, 0]`, faça um laço em `j` subtraindo `fator * U[0, j]` de `U[1, j]`, e atualize `c[1]`. Imprima `U` e `c`.

In [ ]:
# ✍️ passo 3

**Preveja:** quanto vale o fator, e o que aparece em `U[1, 0]`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

O fator é $-5/15 = -0{,}333$. `U[1, 0]` vira **zero**, `U[1, 1]` vira 18,33 e `c[1]` vira 3,33.

</details>

**✍️ Passo 4.** Agora a eliminação inteira, com os três laços do quadro (`k`, `i`, `j`), a partir de `U = A.copy()` e `c = b.copy()`. Imprima `np.round(U, 4)`.

In [ ]:
# ✍️ passo 4

**Preveja:** o que deve aparecer abaixo da diagonal?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Só zeros: `[[15, -5, 0], [0, 18.3333, -10], [0, 0, 19.5455]]`. A matriz é triangular superior.

</details>

> ⚠️ **Armadilha.** O laço de `j` começa em `k`, e não em `0` nem em `k + 1`. Começar em `k + 1`
deixa o "quase zero" abaixo da diagonal sem zerar; a substituição ignora ele e
o resultado sai certo — mas a matriz impressa engana quem confere.

> 🧰 **Python: `range` de trás para frente**
>
> `range(inicio, fim, passo)` com passo **negativo** conta para trás, e o `fim` fica de
> fora como sempre: `range(n - 1, -1, -1)` dá `n-1, ..., 1, 0`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(list(range(2, -1, -1)))

**✍️ Passo 5.** Faça a substituição: `x = np.zeros(3)` e, com `for i in range(2, -1, -1):`, calcule cada `x[i]` pela fórmula do quadro. Imprima `x`.

In [ ]:
# ✍️ passo 5

**Preveja:** as correntes são positivas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`[0.7907 0.3721 0.3488]` A: todas positivas, no sentido escolhido para as
malhas. A malha 0, que tem a fonte maior, leva a maior corrente.

📖 [capítulo 8 · No quadro: a eliminação de Gauss](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#no-quadro-a-eliminacao-de-gauss)

</details>

### 🎯 Sua vez — Deu triangular?

Escreva `eh_triangular(U)`, que devolve `True` se todos os elementos **abaixo** da diagonal têm valor absoluto menor que $10^{-12}$, e `False` caso contrário.

In [ ]:
def eh_triangular(U):
    # sua solução aqui
    pass

In [ ]:
confere(eh_triangular, [
    ((np.array([[2.0, 1.0], [0.0, 3.0]]),), True),
    ((np.array([[2.0, 1.0], [1e-3, 3.0]]),), False),
    ((np.array([[1.0, 2.0, 3.0], [0.0, 4.0, 5.0], [0.0, 1e-15, 6.0]]),), True),
])

<details>
<summary><b>💡 Dica</b></summary>

Dois laços: `for i in range(n)` e `for j in range(i)` (só as colunas antes da diagonal). Um `return False` ao achar um elemento grande; `return True` no fim.

</details>

## 3. Pivotamento

A eliminação **divide pelo pivô**. Se ele for zero, quebra; se for minúsculo, erra. A
célula 📦 tem a sua eliminação, sem pivotamento, escrita como função.

📖 [capítulo 8 · Pivotamento](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#pivotamento)

In [ ]:
# 📦 dados prontos — só rode esta célula
# Gauss SEM pivotamento, escrito como função (é o código dos passos 3 a 5).
def gauss_sem_pivo(A, b):
    A = A.copy()
    b = b.copy()
    n = len(b)
    for k in range(n - 1):
        for i in range(k + 1, n):
            fator = A[i, k] / A[k, k]
            for j in range(k, n):
                A[i, j] = A[i, j] - fator * A[k, j]
            b[i] = b[i] - fator * b[k]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = b[i]
        for j in range(i + 1, n):
            soma = soma - A[i, j] * x[j]
        x[i] = soma / A[i, i]
    return x

**✍️ Passo 6.** Resolva com `gauss_sem_pivo` o sistema `[[1e-17, 1.0], [1.0, 1.0]]` com lado direito `[1.0, 2.0]`, e depois o mesmo sistema com as duas linhas trocadas.

In [ ]:
# ✍️ passo 6

**Preveja:** a resposta certa é $(1, 1)$. As duas versões acertam?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Com o pivô $10^{-17}$ em cima, sai $x = 0$: errado, sem aviso. O fator
$10^{17}$ apagou o "1" da segunda linha no arredondamento. Com as linhas
trocadas (pivô 1), sai $(1, 1)$. A regra: antes de cada coluna, traga para o
pivô a linha com o **maior** valor absoluto.

📖 [capítulo 8 · Pivotamento](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#pivotamento)

</details>

### 🎯 Sua vez — Quem deve ser o pivô?

Escreva `linha_do_pivo(A, k)`, que devolve o índice da linha, de `k` até a última, com o **maior valor absoluto** na coluna `k`.

In [ ]:
def linha_do_pivo(A, k):
    # sua solução aqui
    pass

In [ ]:
M = np.array([[1e-17, 1.0, 2.0], [1.0, 1.0, 1.0], [-3.0, 0.0, 1.0]])
confere(linha_do_pivo, [
    ((M, 0), 2),
    ((M, 1), 1),
    ((np.array([[5.0, 1.0], [1.0, 1.0]]), 0), 0),
])

<details>
<summary><b>💡 Dica</b></summary>

Padrão extremo: comece com `p = k` e percorra as linhas `k + 1` em diante, comparando `abs(A[i, k])` com `abs(A[p, k])`.

</details>

## 4. Confira com a biblioteca

📖 [capítulo 8 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#confira-com-a-biblioteca)

O `np.linalg.solve` da aula 07 faz a eliminação que você acabou de programar, **com
pivotamento**, em código otimizado. E `A @ x` refaz o lado esquerdo: conferir que ele
devolve `b` verifica qualquer resposta.

**✍️ Passo 7.** Resolva o circuito com `np.linalg.solve(A, b)` e imprima `b - A @ x`.

In [ ]:
# ✍️ passo 7

**Preveja:** o resíduo vai dar exatamente zero?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Quase: números da ordem de $10^{-16}$ — arredondamento. As correntes batem
com as do passo 5.

📖 [capítulo 8 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#confira-com-a-biblioteca)

</details>

## 5. Sistemas mal condicionados

📖 [capítulo 8 · Sistemas mal condicionados](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#sistemas-mal-condicionados)

**✍️ Passo 8.** Resolva `x + y = 2`, `x + 1.0001 y = 2.0001` e depois o mesmo sistema com o lado direito `[2.0, 2.0002]` (um erro de 0,005 % num dado).

In [ ]:
# ✍️ passo 8

**Preveja:** a resposta muda muito?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

De $(1, 1)$ para $(0, 2)$: um erro minúsculo nos dados virou uma resposta
completamente diferente. As duas retas são quase paralelas. O sistema é **mal
condicionado**, e nenhum método conserta: a resposta muda de verdade.

📖 [capítulo 8 · Sistemas mal condicionados](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#sistemas-mal-condicionados)

</details>

## 6. Mesmo método, outra área

**Nutrição.** Arroz, feijão e frango têm, por grama, (0,025; 0,28; 0,002),
(0,048; 0,14; 0,005) e (0,32; 0; 0,025) gramas de proteína, carboidrato e gordura.
Quantos gramas de cada dão 50,6 g de proteína, 77 g de carboidrato e 4,15 g de
gordura?

📖 [capítulo 8 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#mesmo-metodo-outra-area)

**✍️ Passo 9.** Monte a matriz com uma **linha por nutriente** e uma **coluna por alimento**, o lado direito com as metas, e resolva.

In [ ]:
# ✍️ passo 9

**Preveja:** a resposta faz sentido como refeição?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

200 g de arroz, 150 g de feijão e 120 g de frango: uma refeição de verdade. Com
metas ao acaso, a resposta pode sair com gramas **negativos** — matemática
certa, prato impossível.

📖 [capítulo 8 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

Retoma o bloco *4. Confira com a biblioteca*.
📖 [capítulo 8 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-eliminacao-gauss/#confira-com-a-biblioteca)

### 🎯 Sua vez — O resíduo

Escreva `residuo_maximo(A, x, b)`, que devolve o **maior valor absoluto** de `b - A @ x`.

In [ ]:
def residuo_maximo(A, x, b):
    # sua solução aqui
    pass

In [ ]:
M = np.array([[2.0, 1.0], [1.0, 3.0]])
v = np.array([3.0, 5.0])
confere(residuo_maximo, [
    ((M, np.array([0.8, 1.4]), v), 0.0),
    ((M, np.array([1.0, 1.0]), v), 1.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Calcule `r = b - A @ x` e aplique o padrão extremo com `abs`.

</details>

## 🧩 Resolvendo o problema

> *"**Onde está a pessoa?**"* — a central de emergência.

Com a torre $k$ em $(x_k, y_k)$ e a distância $r_k$, o celular está no círculo
$(x - x_k)^2 + (y - y_k)^2 = r_k^2$. Subtraindo a equação da torre 1 da equação da
torre $k$, os termos $x^2$ e $y^2$ se cancelam e sobra uma equação **linear**:

$$ 2(x_k - x_1)\,x + 2(y_k - y_1)\,y = r_1^2 - r_k^2 + x_k^2 - x_1^2 + y_k^2 - y_1^2. $$

Com as torres 2 e 3, são duas equações: um sistema 2 × 2.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Três torres de celular (posições em km) e a distância estimada do aparelho a
# cada uma, pelo tempo que o sinal leva para ir e voltar.
torres = np.array([[0.0, 0.0],
                   [4.0, 0.0],
                   [0.0, 3.0]])
distancias = np.array([1.921, 2.773, 2.343])

### 🎯 Sua vez — A posição do celular

Escreva `posicao(torres, distancias)`, que monta o sistema 2 × 2 acima (use a
torre 0 como referência, e as torres 1 e 2 para as duas equações) e devolve a
posição `[x, y]` com `np.linalg.solve`. Crie a matriz com `np.zeros((2, 2))`,
uma matriz 2 × 2 de zeros.

In [ ]:
def posicao(torres, distancias):
    # sua solução aqui
    pass

In [ ]:
confere(posicao, [
    ((torres, distancias), [1.500089, 1.2000986666666666]),
], tol=1e-9)

<details>
<summary><b>💡 Dica</b></summary>

`x1, y1 = torres[0]` separa as duas coordenadas da torre 0. Num laço com `k` de
0 a 1, a torre usada é `torres[k + 1]`, e a linha `k` da matriz recebe os dois
coeficientes da fórmula.

</details>

A resposta para a central, conferida nas três distâncias:

In [ ]:
p = posicao(torres, distancias)
if p is not None:
    print("posição (km):", p)
    for k in range(3):
        dx = p[0] - torres[k, 0]
        dy = p[1] - torres[k, 1]
        print("torre", k, ": distância medida", distancias[k], "| distância da resposta", np.sqrt(dx**2 + dy**2))

<details>
<summary><b>▶ O que os números dizem</b></summary>

O celular está em **(1.500; 1.200) km**, e as distâncias da resposta
às três torres batem com as medidas até a terceira casa.

Na prática, as distâncias têm erro (o sinal reflete em prédios), e com **mais** de
três torres o sistema tem mais equações que incógnitas: não há ponto que satisfaça
todas. A saída é achar o ponto que **erra menos** em todas elas — os mínimos
quadrados da Unidade 7.

</details>

## 📋 A lista

Abra a [Lista 08](https://lacouth.github.io/metodos_telecom-site/listas/lista08/). O **Exercício 01** é à mão (✏️): uma eliminação 3 × 3.
Comece por ele, no papel.

**a)** Qual o fator para zerar o $-3$ da segunda linha, usando a primeira?

<details>
<summary><b>▶ Resposta</b></summary>

$-3/2 = -1{,}5$. A linha 1 fica: linha 1 $-(-1{,}5)\times$ linha 0.

</details>

Termine o exercício e siga para o **Exercício 02**, a eliminação como função.

## 🚪 Antes de sair

**1.** Por que a eliminação de Gauss precisa trabalhar numa **cópia** da matriz?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque ela modifica a matriz (zera abaixo da diagonal). Sem a cópia, a matriz original de quem chamou a função sairia estragada.

</details>

**2.** O que é o pivô, e por que um pivô minúsculo é perigoso?

<details>
<summary><b>▶ Resposta da 2</b></summary>

É o elemento da diagonal que divide na eliminação. Dividir por um número minúsculo gera fatores enormes, que apagam os outros números no arredondamento.

</details>

**3.** Um sistema mal condicionado se resolve com um método melhor?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Não: o problema está nos dados. Pequenas mudanças neles mudam a resposta de verdade. O que se faz é medir melhor ou reformular o problema.

</details>

## 🏠 Para casa

- Refaça no papel a eliminação 3 × 3 do quadro **sem olhar**.
- Termine a [Lista 08](https://lacouth.github.io/metodos_telecom-site/listas/lista08/).
- Leia o começo do [capítulo 9](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-jacobi-gauss-seidel/):
  e se o sistema tiver um milhão de equações?